# Updating and Deleting Data

So far you have added data and read it. Now you will learn to change existing
rows with **UPDATE** and remove rows with **DELETE**. You will also grow your
Library: by the end of this notebook, each book will track whether it is
available to borrow.

## A safety net first

UPDATE and DELETE change your data permanently. If an experiment goes wrong,
you can always restore the original Library data by running
`reset_my_project_data()`. Let's start from a clean state:

In [ ]:
from rel_db_learning import get_project_connection, run_sql, run_ddl, reset_my_project_data

project = get_project_connection()
reset_my_project_data()

## UPDATE: changing existing rows

UPDATE changes values in rows that match a condition:

```sql
UPDATE table SET column = value WHERE condition;
```

The WHERE clause is critical. Without it, UPDATE changes every row. Let's
look at one loan before changing it:

In [ ]:
run_sql("SELECT loan_id, loan_date, return_date FROM loan WHERE loan_id = 1", project)

Suppose loan 1 was returned today. We set its return_date:

In [ ]:
run_ddl("UPDATE loan SET return_date = '2026-06-04' WHERE loan_id = 1", project)
run_sql("SELECT loan_id, loan_date, return_date FROM loan WHERE loan_id = 1", project)

Always include a WHERE clause unless you truly mean to change every row. This
sets the same return_date on every loan, which is almost never what you want:

```sql
UPDATE loan SET return_date = '2026-06-04';  
```

## DELETE: removing rows

DELETE removes rows that match a condition. Like UPDATE, it needs a WHERE
clause, or it removes everything.

```sql
DELETE FROM table WHERE condition;
```

Let's count loans, delete one specific loan, then count again:

In [ ]:
run_sql("SELECT COUNT(*) AS loan_count FROM loan", project)

In [ ]:
run_ddl("DELETE FROM loan WHERE loan_id = 50", project)
run_sql("SELECT COUNT(*) AS loan_count FROM loan", project)

One row fewer. If you ever delete or change something by mistake, restore the
full dataset:

In [ ]:
reset_my_project_data()

## Growing the Library: tracking availability

Right now there is no way to tell whether a book is currently available to
borrow. We will add a new column, `is_available` (a BOOLEAN), to the book
table. As always, we design the change in pgModeler.

One thing first. Your database already has data. pgModeler's normal export produces the full CREATE TABLE for the final design, which cant run against a table that already exists. Instead we use pgModeler's **Diff**
feature, which compares your model to the live database and generates exactly
the change needed (an ALTER TABLE), leaving your data intact. This is how
schema changes are applied to real, populated databases.

### Step 1: Add the column in the model

1. Open your `stage_1_initial_design.dbm` file in pgModeler (this is your
   current Library design).
2. Double-click the **book** table to edit it.
3. Add a new column:
   - Name: `is_available`
   - Type: `boolean`
4. Use **File > Save As** and save the model as `stage_3_design.dbm` in your
   `student_work/my_project/` folder. From now on, each schema stage gets its
   own `.dbm` file, so you keep a clear history of how the Library evolved.

### Step 2: Connect pgModeler to your database (one time)

The Diff feature needs a connection to your live database. If you have not
set one up in pgModeler yet:

1. Open the connections manager (the database/plug icon in the toolbar, or
   **Edit > Connections** depending on your pgModeler version).
2. Add a connection with the same details you use for my_project:
   - Host: `localhost`
   - Port: `5432`
   - User and password: your PostgreSQL credentials
   - You can leave the database blank or set it to `my_project`.
3. Save and test the connection.

You only need to do this once. pgModeler remembers the connection for future
stages.

### Step 3: Run the Diff

1. Open pgModeler's **Diff** tool in the left tool bar.
2. Set the comparison so that:
   - The **source** is your current model (`stage_3_design.dbm`).
   - The **target / compare-against** is your live `my_project` database
     (select the connection from Step 2, and the `my_project` database).
3. Generate the diff. pgModeler will produce the SQL needed to bring the
   database in line with your model. With only the new column added, this
   should be a single statement similar to:

   `ALTER TABLE public.book ADD COLUMN is_available boolean;`

4. Review the generated SQL. It should only ADD the is_available column. If
   it proposes dropping anything, stop and check that your model still
   matches the rest of the schema. Copy the ALTER statement.

### Step 4: Apply the change

Paste the ALTER statement that pgModeler's Diff generated into the cell
below, then run it.

In [ ]:
# Paste the ALTER statement that pgModeler's Diff generated, between the
# quotes. With only the new column added, it should be a single ALTER TABLE
# that adds is_available to the book table.
alter_sql = """
-- Paste the ALTER TABLE statement from pgModeler's Diff here, for example:
-- ALTER TABLE public.book ADD COLUMN is_available boolean;
"""

run_ddl(alter_sql, project)

### Step 4: Validate the change

Let's confirm the book table now has the is_available column, by validating
against the live database.

In [ ]:
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_STAGE_3_AVAILABILITY

validate_exercise(
    project_db=get_project_connection(),
    spec=PROJECT_STAGE_3_AVAILABILITY,
)

## Using the new column

New columns start empty (NULL). Let's set every book to available as a
starting point, then mark a few as unavailable.

In [ ]:
run_ddl("UPDATE book SET is_available = true", project)
run_sql("SELECT book_id, title, is_available FROM book ORDER BY book_id LIMIT 10", project)

Now suppose two books are currently out and not available. We mark them:

In [ ]:
run_ddl("UPDATE book SET is_available = false WHERE book_id IN (1, 2)", project)
run_sql("SELECT book_id, title, is_available FROM book WHERE is_available = false", project)

## Exercise

Let's practice UPDATE on a clean, known state. First run the setup cell
below, which marks every book as available. Then write your own UPDATE, run
it, and check the result.

In [ ]:
# Setup: start from a clean state where every book is available.
run_ddl("UPDATE book SET is_available = true", project)
run_sql("SELECT COUNT(*) AS available_books FROM book WHERE is_available = true", project)

**Your task:** Write an UPDATE that marks the book titled 'The Silmarillion'
as unavailable (is_available = false). Match the book by its title in the
WHERE clause. Write your statement below, then run the cell.

In [ ]:
# Write your UPDATE statement here, then run this cell.
my_update = "??? "
run_ddl(my_update, project)

Now check your work. If your UPDATE was correct, exactly one book, The
Silmarillion, should be unavailable.

In [ ]:
from rel_db_learning.widgets.self_assessment import sql_query_check

sql_query_check(
    student_sql="SELECT title FROM book WHERE is_available = false",
    correct_sql="m04_nb_08_exercise_1",
    connection=project,
)

If you want to return everything to the original Library data, run
`reset_my_project_data()`. It restores the core data, and is_available goes
back to its default.

## What's next

You can now change and remove data, and your Library tracks availability.
Next you will learn **Common Table Expressions** (CTEs) with the WITH clause,
a way to build readable, multi-step queries.